# Florida Inspection Data Cleaning

**Area:** Hillsborough, Pinellas, and Pasco counties (Tampa Bay).  
**Dates:** July 1, 2021–June 30, 2022.

Put `fdinspi_2122.xlsx` in the same folder as this notebook. In Positron, select a Python interpreter and run the cells in order. The Excel file may take several minutes to load.

This notebook cleans Florida records and creates one summary row per restaurant license. Files are saved in `florida_cleaned_outputs`.

In [ ]:
# If needed, uncomment and run this line once:
# %pip install pandas openpyxl

## 1. Set the file location and study area

In [ ]:
from pathlib import Path
import pandas as pd

# If needed, replace with the full path to your Excel file.
# Windows example: Path(r"C:\Users\alexa\Downloads\fdinspi_2122.xlsx")
FL_FILE = Path.cwd() / "fdinspi_2122.xlsx"
OUT = FL_FILE.parent / "florida_cleaned_outputs"
OUT.mkdir(exist_ok=True)
BASE = FL_FILE.parent
COUNTIES = ["HILLSBOROUGH", "PINELLAS", "PASCO"]
START = pd.Timestamp("2021-07-01")
END = pd.Timestamp("2022-06-30")


def save(frame, filename):
    frame.to_csv(OUT / filename, index=False)
    print(f"SAVED: {filename} — {len(frame):,} rows")


def zip5(series):
    return series.astype("string").str.strip().str.extract(r"^(\d{5})", expand=False)


## 2. Load and clean the columns

Uses the actual headers in your file. Converts dates and violation counts; missing counts remain missing.

In [ ]:
FL_COLUMNS = {
    'CNTY_DESC': 'county_name', 'LICENSE_NO': 'license_number',
    'DBA_NAME': 'name', 'LOC_ADDRESS': 'address', 'LOC_CITY': 'city',
    'LOC_ZIP': 'zip', 'INSP_NO': 'inspection_number', 'VISIT_NO': 'visit_number',
    'INSPTYPE': 'inspection_type', 'DISPOSITION': 'disposition',
    'INSP_DATE': 'inspection_date', 'VIOLATIONS': 'total_violations',
    'HIGH_VIOL': 'high_priority', 'INTERMED_VIOL': 'intermediate',
    'BASIC_VIOL': 'basic', 'LIC_ID': 'license_id', 'INSP_VST_ID': 'inspection_visit_id'
}
if not FL_FILE.exists():
    raise FileNotFoundError(f'Put fdinspi_2122.xlsx in {BASE}, or edit FL_FILE.')

print('Loading Florida Excel file; the statewide file can take several minutes.')
fl_all = pd.read_excel(FL_FILE, usecols=list(FL_COLUMNS), dtype='string')
fl_all = fl_all.rename(columns=FL_COLUMNS)
print('FLORIDA INPUT SHAPE:', fl_all.shape)
print(fl_all.head().to_string(index=False))
print('\nINSPECTION TYPES:\n', fl_all['inspection_type'].value_counts().to_string())

for col in ['county_name', 'inspection_type', 'disposition', 'license_number']:
    fl_all[col] = fl_all[col].str.strip()
fl_all['inspection_date'] = pd.to_datetime(fl_all['inspection_date'], errors='coerce')
for col in ['visit_number', 'total_violations', 'high_priority', 'intermediate', 'basic']:
    fl_all[col] = pd.to_numeric(fl_all[col], errors='coerce')
fl_all['zip'] = zip5(fl_all['zip'])
print('ACTUAL FILE DATES:', fl_all['inspection_date'].min(), 'to',
      fl_all['inspection_date'].max())

## 3. Keep initial routine inspections

Keeps the three counties and study dates. VISIT_NO = 1 removes callbacks, which can also be labeled Routine - Food. Failed initial inspections remain included. Audit CSVs flag duplicate IDs, missing IDs and inconsistent violation totals.

In [ ]:
fl_tampa_all_visits = fl_all[
    fl_all['county_name'].str.upper().isin(COUNTIES)
    & fl_all['inspection_date'].between(START, END)
].copy()
save(fl_tampa_all_visits['inspection_type'].value_counts().rename_axis(
    'inspection_type').reset_index(name='n_visits'), 'fl_inspection_type_counts.csv')

# INSPTYPE alone does NOT remove callbacks: some retain 'Routine - Food'.
# VISIT_NO == 1 selects the initial visit. Disposition is an additional guard.
# Keep failed initial visits; excluding them would bias the violation averages.
callback = fl_tampa_all_visits['disposition'].str.contains(
    r'call\s*back|callback|re[- ]?inspect', case=False, regex=True, na=False)
fl_routine = fl_tampa_all_visits[
    fl_tampa_all_visits['inspection_type'].str.casefold().eq('routine - food')
    & fl_tampa_all_visits['visit_number'].eq(1)
    & ~callback
].copy()
save(fl_routine[fl_routine['inspection_visit_id'].isna()], 'fl_missing_visit_ids.csv')
with_ids = fl_routine.dropna(subset=['inspection_visit_id'])
duplicates = with_ids[with_ids.duplicated('inspection_visit_id', keep=False)]
save(duplicates, 'fl_duplicate_visit_ids.csv')
conflicts = duplicates.drop_duplicates().duplicated('inspection_visit_id').any()
if conflicts:
    raise ValueError('Conflicting rows share an inspection visit ID; review the audit CSV.')
fl_routine = pd.concat([
    with_ids.drop_duplicates('inspection_visit_id'),
    fl_routine[fl_routine['inspection_visit_id'].isna()].drop_duplicates()
], ignore_index=True)
save(fl_routine[fl_routine['license_number'].isna()], 'fl_missing_license_numbers.csv')
fl_routine = fl_routine.dropna(subset=['license_number']).sort_values('inspection_date')

# Preserve missing violation counts as missing: they do not mean zero.
counts = ['total_violations', 'high_priority', 'intermediate', 'basic']
bad = fl_routine[counts].lt(0).any(axis=1)
if bad.any():
    raise ValueError('Negative violation counts found; review source rows.')
complete = fl_routine[counts].notna().all(axis=1)
sum_counts = fl_routine[['high_priority', 'intermediate', 'basic']].sum(axis=1)
save(fl_routine[complete & sum_counts.ne(fl_routine['total_violations'])],
     'fl_violation_count_discrepancies.csv')
save(fl_routine, 'fl_tampa_routine_inspections.csv')

## 4. Create one row per restaurant license

Counts routine inspections and calculates average total, high-priority, intermediate and basic violations. Identity fields come from the most recent included visit. Licenses with multiple names or addresses are flagged for review.

In [ ]:
fl_summary = fl_routine.groupby('license_number', as_index=False).agg(
    n_inspections=('inspection_date', 'size'),
    n_scored_inspections=('total_violations', 'count'),
    first_inspection=('inspection_date', 'min'),
    last_inspection=('inspection_date', 'max'),
    avg_violations=('total_violations', 'mean'),
    max_violations=('total_violations', 'max'),
    avg_high_priority=('high_priority', 'mean'),
    avg_intermediate=('intermediate', 'mean'),
    avg_basic=('basic', 'mean')
)
# Take identity fields from a single most recent row, not unrelated first values.
identity = fl_routine.drop_duplicates('license_number', keep='last')[
    ['license_number', 'license_id', 'name', 'address', 'city', 'zip', 'county_name']]
fl_summary = identity.merge(fl_summary, on='license_number', validate='one_to_one')
location_changes = fl_routine.groupby('license_number').agg(
    n_addresses=('address', 'nunique'), n_names=('name', 'nunique'))
save(location_changes[location_changes.gt(1).any(axis=1)].reset_index(),
     'fl_identity_changes_review.csv')
assert fl_summary['license_number'].is_unique
assert fl_summary['n_inspections'].sum() == len(fl_routine)
save(fl_summary, 'fl_restaurant_inspection_summary.csv')

## 5. View the cleaned summary

Main files:
- `fl_tampa_routine_inspections.csv`: one row per included inspection visit.
- `fl_restaurant_inspection_summary.csv`: one row per restaurant license.

A license does not automatically correspond to one Yelp business. Review `fl_identity_changes_review.csv` before joining another dataset.

In [ ]:
print(f"CLEANED ROUTINE INSPECTIONS: {len(fl_routine):,}")
print(f"RESTAURANT LICENSES: {len(fl_summary):,}")
print(f"OUTPUT FOLDER: {OUT.resolve()}")
display(fl_summary.head(10))